## Imports

In [ ]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../../'))

import neuro_fuzzy_toolbox as nft

In [7]:
import numpy as np

In [8]:
from sklearn.preprocessing import MinMaxScaler

# Surface (1k)

## Data

In [9]:
def z(x, y):
  return ((3) * ((1-x)**2) * (np.exp(-(x**2)-((y+1)**2))) - (10) * ((x/5)-(x**3)-(y**5)) * (np.exp(-(x**2)-(y**2))) - (1/3)*np.exp(-(x+1)**2-(y**2)))

#Training
x0 = np.random.uniform(-3,3,1000)
x1 = np.random.uniform(-3,3,1000)

e = np.random.normal(0,0.5,1000) #noise
Y = z(x0,x1) + e

#Testing
x0_test = np.random.uniform(-3,3,1000)
x1_test = np.random.uniform(-3,3,1000)

Y_test = z(x0_test,x1_test)

In [10]:
#Training
scaler = MinMaxScaler(feature_range=(0, 1))
vstack_train = np.vstack((x0,x1)).T
scaled_train = scaler.fit_transform(vstack_train)

#Testing
vstack_test = np.vstack((x0_test,x1_test)).T
scaled_test = scaler.transform(vstack_test)

In [11]:
dtype = torch.float64

train_loader = data.DataLoader(
    data.TensorDataset(
        torch.tensor(scaled_train, dtype=dtype), 
        torch.tensor(Y, dtype=dtype)), 
    batch_size = 8,
    shuffle = True)

x_train = train_loader.dataset.tensors[0]
y_train = train_loader.dataset.tensors[1]

x_test = torch.tensor(scaled_test, dtype=dtype)
y_test = torch.tensor(Y_test, dtype=dtype)

In [12]:
x_train.dtype

torch.float64

## Model & Training

### ANFIS

In [40]:
model = nft.rule_reduced_ANFIS(
    input_size = 2,
    num_mfs = 1,
    outputs = 1,
    dtype=torch.float64
)

In [41]:
model.init_premises(x_train)

### Hybrid Learning Algorithm

In [42]:
loss_fn = nn.functional.mse_loss
#loss_fn = nn.functional.binary_cross_entropy
#loss_fn = nn.functional.cross_entropy

optimizer = torch.optim.AdamW
params = {'lr': 0.001, 'weight_decay': 0.001}

early_stopping = nft.EarlyStopping(patience=4)

In [43]:
trainer = nft.Hybrid_learning_algorithm(
    epochs=20,
    loss_function=loss_fn,
    optimizer=optimizer,
    optimizer_params=params,
    early_stopping=early_stopping
)

### SONFIS

In [44]:
Ngrow = 40
dGrow = 0.8
Nsplit = 20
eSplit = 0.5
Nvanish = 6
lVanish = 4

max_iterations = 40

anfis_trainer = trainer

validation = 0.2
sonfis_early_stopping = nft.EarlyStopping(patience=7)
last_training_iteration = True

In [45]:
sonfis = nft.SONFIS(
    Ngrow=Ngrow,
    dGrow=dGrow,
    Nsplit=Nsplit,
    eSplit=eSplit,
    Nvanish=Nvanish,
    lVanish=lVanish,
    max_iterations=max_iterations,
    ANFIStrainer=anfis_trainer,
    validation=validation,
    early_stopping=sonfis_early_stopping,
    last_training_iteration=last_training_iteration
)

In [46]:
%time sonfis(model, train_loader, verbose=True)

Iteration:  0/40 - loss: 3.948072 - validation loss: 3.518710
 -> ANFIS rules: 1

Iteration:  1/40 - loss: 4.779079 - validation loss: 4.291391
 -> ANFIS rules: 2

Iteration:  2/40 - loss: 20.825201 - validation loss: 19.236445
 -> ANFIS rules: 3

Iteration:  3/40 - loss: 22.355117 - validation loss: 21.046140
 -> ANFIS rules: 5

Iteration:  4/40 - loss: 10.253559 - validation loss: 10.076487
 -> ANFIS rules: 7

Iteration:  5/40 - loss: 11.592401 - validation loss: 11.048685
 -> ANFIS rules: 7

Iteration:  6/40 - loss: 0.630478 - validation loss: 0.662279
 -> ANFIS rules: 13

Iteration:  7/40 - loss: 0.620781 - validation loss: 0.652924
 -> ANFIS rules: 15

Iteration:  8/40 - loss: 0.621765 - validation loss: 0.656501
 -> ANFIS rules: 16

Iteration:  9/40 - loss: 4.764628 - validation loss: 3.939423
 -> ANFIS rules: 20

Iteration: 10/40 - loss: 5.173718 - validation loss: 4.073785
 -> ANFIS rules: 26

Iteration: 11/40 - loss: 5.058339 - validation loss: 3.948143
 -> ANFIS rules: 27

It

In [47]:
test_measures = nft.get_measures(model, x_test, y_test)

for measure in test_measures:
    print(measure + ':', test_measures[measure])

MSE: 0.15575213069144156
RMSE: 0.3946544446619619
MAE: 0.2804268145207111
R2: 0.9549196427397557
MAPE: 37.651067871258334


In [48]:
train_measures = nft.get_measures(model, x_train, y_train)

for measure in train_measures:
    print(measure + ':', train_measures[measure])

MSE: 0.36890845569452685
RMSE: 0.6073783464155821
MAE: 0.47765657629109903
R2: 0.8932710242329939
MAPE: 1.6536337873850089
